# 1. Data Preprocessing

This notebook prepares our raw datasets (`SEntFiN-v1.1.csv` and `Loughran-McDonald_MasterDictionary_1993-2024.csv`) for the sentiment analysis pipeline.

**Goals:**
1. Clean the Kaggle aspect-based sentiment dataset (SEntFiN) into a format ready for sentence-level models.
2. Filter the massive Loughran-McDonald dictionary to only include relevant sentiment-bearing words.

In [2]:
import pandas as pd
import ast
import os

RAW_DATA_DIR = '../raw_data'
PROCESSED_DATA_DIR = '../data/processed'
os.makedirs(PROCESSED_DATA_DIR, exist_ok=True)

## 1. Process SEntFiN (Kaggle Financial News)
The dataset contains headlines, entity-sentiment mappings (in a string dict format), and extra noise columns.

In [3]:
sentfin_path = os.path.join(RAW_DATA_DIR, 'SEntFiN-v1.1.csv')
df_sentfin = pd.read_csv(sentfin_path)
print(f"Original shape: {df_sentfin.shape}")

# Drop unnecessary columns
df_sentfin = df_sentfin.drop(columns=['S No.', 'Words'], errors='ignore')

# Parse the 'Decisions' column from string representations of dictionaries to actual dictionaries
df_sentfin['Decisions'] = df_sentfin['Decisions'].apply(ast.literal_eval)
df_sentfin.head()

Original shape: (10753, 4)


,Title,Decisions
0,SpiceJet to issue 6.4 crore warrants to promoters,{'SpiceJet': 'neutral'}
1,MMTC Q2 net loss at Rs 10.4 crore,{'MMTC': 'neutral'}
2,"Mid-cap funds can deliver more, stay put: Experts",{'Mid-cap funds': 'positive'}
3,Mid caps now turn into market darlings,{'Mid caps': 'positive'}
4,"Market seeing patience, if not conviction: Pra...",{'Market': 'neutral'}


### Handling Multi-Entity Sentiments
Some headlines contain multiple entities (e.g., Apple goes up, Samsung goes down). 
For our Lexicon and Classical ML baselines, sentence-level sentiment works best when there is a **single dominant sentiment**. 
Let's create two versions of the dataset:
1. **Strict dataset:** Only keep headlines where all entities share the same sentiment (no conflict).
2. **Exploded dataset:** Extract each entity-sentiment pair as its own row (useful for aspect-based models later if needed).

In [4]:
strict_rows = []
exploded_rows = []

for _, row in df_sentfin.iterrows():
    text = row['Title']
    decisions = row['Decisions']
    
    # For exploded dataset
    for entity, sentiment in decisions.items():
        exploded_rows.append({'text': text, 'entity': entity, 'sentiment': sentiment})
        
    # For strict dataset
    unique_sentiments = set(decisions.values())
    if len(unique_sentiments) == 1:
        strict_rows.append({'text': text, 'sentiment': list(unique_sentiments)[0]})

df_strict = pd.DataFrame(strict_rows)
df_exploded = pd.DataFrame(exploded_rows)

print(f"Strict dataset shape: {df_strict.shape}")
print(f"Exploded dataset shape: {df_exploded.shape}")

# Save them
df_strict.to_csv(os.path.join(PROCESSED_DATA_DIR, 'sentfin_strict.csv'), index=False)
df_exploded.to_csv(os.path.join(PROCESSED_DATA_DIR, 'sentfin_exploded.csv'), index=False)

Strict dataset shape: (9518, 2)
Exploded dataset shape: (14409, 3)


## 2. Process Loughran-McDonald Master Dictionary
The raw dictionary is huge and contains many neutral words (financial jargon with no sentiment). We only need words that trigger at least one sentiment flag.

In [5]:
lm_path = os.path.join(RAW_DATA_DIR, 'Loughran-McDonald_MasterDictionary_1993-2024.csv')
df_lm = pd.read_csv(lm_path)
print(f"Original LM Dictionary shape: {df_lm.shape}")

# Sentiment columns in the dataset
sentiment_cols = ['Negative', 'Positive', 'Uncertainty', 'Litigious', 'Strong_Modal', 'Weak_Modal', 'Constraining']

# Filter to only include words where at least one sentiment flag is > 0
df_lm_filtered = df_lm[df_lm[sentiment_cols].sum(axis=1) > 0].copy()
print(f"Filtered LM Dictionary shape: {df_lm_filtered.shape}")

# Keep only necessary columns
cols_to_keep = ['Word'] + sentiment_cols
df_lm_filtered = df_lm_filtered[cols_to_keep]

# Lowercase words for easier matching later
df_lm_filtered['Word'] = df_lm_filtered['Word'].str.lower()

# Save it
df_lm_filtered.to_csv(os.path.join(PROCESSED_DATA_DIR, 'LM_Dictionary_Filtered.csv'), index=False)
df_lm_filtered.head()

Original LM Dictionary shape: (86553, 17)
Filtered LM Dictionary shape: (3857, 17)


,Word,Negative,Positive,Uncertainty,Litigious,Strong_Modal,Weak_Modal,Constraining
9,abandon,2009,0,0,0,0,0,0
10,abandoned,2009,0,0,0,0,0,0
11,abandoning,2009,0,0,0,0,0,0
12,abandonment,2009,0,0,0,0,0,0
13,abandonments,2009,0,0,0,0,0,0


Processing complete. The clean datasets are now available in `data/processed/` and ready for the modeling phases.